# SMS Spam: Exploratory Data Analysis

Load the local dataset through the same normalization helper used by training, then inspect label balance, message lengths, and frequent terms. Run Jupyter from the project root.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from src.preprocess import clean_text, load_dataset

data_path = Path('data/scam.csv')
dataset = load_dataset(data_path)
dataset.head()

## Label balance

Check the class counts and proportions before modeling.

In [ ]:
label_counts = dataset['label'].value_counts()
display(pd.DataFrame({'count': label_counts, 'fraction': label_counts / len(dataset)}))
label_counts.plot(kind='bar', color=['#4c8c72', '#d65a4a'])
plt.title('Message labels')
plt.xlabel('Label')
plt.ylabel('Messages')
plt.tight_layout()
plt.show()

## Message length

Compare character counts by class to spot useful distribution differences and outliers.

In [ ]:
dataset['character_count'] = dataset['message'].str.len()
display(dataset.groupby('label')['character_count'].describe())
dataset.boxplot(column='character_count', by='label', grid=False)
plt.title('Message length by label')
plt.suptitle('')
plt.xlabel('Label')
plt.ylabel('Characters')
plt.tight_layout()
plt.show()

## Frequent terms

Inspect the most common normalized terms for each class. These raw frequencies are descriptive, not measures of predictive importance.

In [ ]:
def frequent_terms(messages, count=15):
    vectorizer = CountVectorizer(stop_words='english', preprocessor=clean_text)
    matrix = vectorizer.fit_transform(messages)
    totals = matrix.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    order = totals.argsort()[::-1][:count]
    return pd.DataFrame({'term': terms[order], 'count': totals[order]})

for label in ['ham', 'spam']:
    print(f'Top terms in {label}:')
    display(frequent_terms(dataset.loc[dataset['label'] == label, 'message']))